# Project Introduction

Business Context:
This project analyzes an online retail business to understand revenue performance, customer behaviour, product demand, and opportunities for improvement.

Business Goal:
Help the company answer: Where is revenue coming from? Which customers and products matter most? Are customers returning? Which countries or products should the business focus on?

SQL Skills Used:
SELECT, WHERE, GROUP BY, ORDER BY, aggregate functions, date functions, CASE WHEN, subqueries, joins if you split the data, and basic segmentation.

Installing SQL

In [ ]:
!pip install ipython-sql prettytable==3.9.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 66.9 MB/s eta 0:00:00


In [ ]:
%load_ext sql

In [ ]:
%config SqlMagic.style = 'DEFAULT'

In [ ]:
import pandas as pd

file_path = "/content/Online_Retail[1].xlsx"

df = pd.read_excel(file_path)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
import sqlite3

conn = sqlite3.connect("online_retail.db")

df.to_sql("online_retail", conn, if_exists="replace", index=False)

conn.close()

In [ ]:
%sql sqlite:///online_retail.db

Testing

In [ ]:
%%sql

SELECT *
FROM online_retail
LIMIT 5;

 * sqlite:///online_retail.db
Done.


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# Solving Questions

### What is the total revenue of the business?

In [ ]:
%%sql
SELECT
ROUND(SUM(UnitPrice * Quantity),2) as total_revenue
from online_retail
WHERE Quantity > 0 AND UnitPrice > 0;


 * sqlite:///online_retail.db
Done.


total_revenue
10666684.54


### How many unique customers placed orders?

In [ ]:
%%sql
SELECT
COUNT(DISTINCT(CustomerId)) as unique_customers
from online_retail
WHERE CustomerId is not NULL;

 * sqlite:///online_retail.db
Done.


unique_customers
4372


### What is the average order value?

In [ ]:
%%sql
SELECT
ROUND(AVG(order_revenue), 2) as average_order_value
FROM (
  SELECT
  InvoiceNo,
  SUM(Quantity * UnitPrice) as order_revenue
  FROM online_retail
  WHERE Quantity > 0 AND UnitPrice > 0
  GROUP BY InvoiceNo
) as order_table;


 * sqlite:///online_retail.db
Done.


average_order_value
534.4


### Which months generated the highest revenue?

In [ ]:
%%sql
SELECT
strftime('%Y-%m', InvoiceDate) AS month,
ROUND(SUM(UnitPrice * Quantity),2) as monthly_revenue
from online_retail
WHERE Quantity>0 AND UnitPrice>0
GROUP BY month
ORDER BY month;



 * sqlite:///online_retail.db
Done.


month,monthly_revenue
2010-12,823746.14
2011-01,691364.56
2011-02,523631.89
2011-03,717639.36
2011-04,537808.62
2011-05,770536.02
2011-06,761739.9
2011-07,719221.19
2011-08,759138.38
2011-09,1058590.17


### Which countries generate the most revenue?

In [ ]:
%%sql
SELECT
country,
ROUND(SUM(UnitPrice * Quantity),2) as total_revenue
FROM online_retail
WHERE country IS NOT NULL
GROUP BY country
ORDER BY total_revenue DESC;



 * sqlite:///online_retail.db
Done.


Country,total_revenue
United Kingdom,8187806.36
Netherlands,284661.54
EIRE,263276.82
Germany,221698.21
France,197403.9
Australia,137077.27
Switzerland,56385.35
Spain,54774.58
Belgium,40910.96
Sweden,36595.91


### Who are the top 10 customers by revenue?

In [ ]:
%%sql
SELECT
CustomerId,
ROUND(SUM(UnitPrice * Quantity),2) as total_revenue
FROM online_retail
WHERE Quantity>0 AND UnitPrice>0 AND CustomerId is not NULL
GROUP BY CustomerId
ORDER BY total_revenue DESC
LIMIT 10;


 * sqlite:///online_retail.db
Done.


CustomerID,total_revenue
14646.0,280206.02
18102.0,259657.3
17450.0,194550.79
16446.0,168472.5
14911.0,143825.06
12415.0,124914.53
14156.0,117379.63
17511.0,91062.38
16029.0,81024.84
12346.0,77183.6


### Are customers mostly one-time buyers or repeat buyers?

In [ ]:
%%sql
SELECT
customer_type,
COUNT(*) as number_of_customers
FROM
  (SELECT
  CustomerId,
  COUNT(DISTINCT InvoiceNo) as number_of_orders,
  CASE
  WHEN COUNT(DISTINCT InvoiceNo) =1 then 'one time customer'
  ELSE 'Repeat customer'
  END AS customer_type
  FROM online_retail
  WHERE QUANTITY > 0 and UnitPrice > 0 AND CustomerId is not NULL
  GROUP BY CustomerId
) as customer_orders
GROUP BY customer_type;


 * sqlite:///online_retail.db
Done.


customer_type,number_of_customers
Repeat customer,2845
one time customer,1493


### Which products generate the most revenue?

In [ ]:
%%sql
SELECT
DISTINCT StockCode,
Description,
ROUND(SUM(UnitPrice*Quantity),2) as total_revenue
FROM online_retail
WHERE UnitPrice>0 AND Quantity >0 AND CustomerId is not NULL
GROUP BY StockCode
ORDER BY total_revenue DESC
LIMIT 10;

 * sqlite:///online_retail.db
Done.


StockCode,Description,total_revenue
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.6
22423,REGENCY CAKESTAND 3 TIER,142592.95
85123A,WHITE HANGING HEART T-LIGHT HOLDER,100603.5
85099B,JUMBO BAG RED RETROSPOT,85220.78
23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73
POST,POSTAGE,77803.96
47566,PARTY BUNTING,68844.33
84879,ASSORTED COLOUR BIRD ORNAMENT,56580.34
M,Manual,53779.93
23084,RABBIT NIGHT LIGHT,51346.2


### Which products sell high quantity but generate low revenue?

In [ ]:
%%sql
SELECT
StockCode,
Description,
SUM(Quantity) as quantity_sold,
ROUND(SUM(UnitPrice*Quantity),2) as total_revenue
FROM online_retail
WHERE Quantity>0 AND UnitPrice>0 AND StockCode is not NULL
GROUP BY StockCode, Description
ORDER BY quantity_sold DESC
LIMIT 10;


 * sqlite:///online_retail.db
Done.


StockCode,Description,quantity_sold,total_revenue
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.6
23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92
84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,55047,13841.85
85099B,JUMBO BAG RED RETROSPOT,48474,94340.05
85123A,WHITE HANGING HEART T-LIGHT HOLDER,37599,104340.29
22197,POPCORN HOLDER,36761,34298.87
84879,ASSORTED COLOUR BIRD ORNAMENT,36461,59094.93
21212,PACK OF 72 RETROSPOT CAKE CASES,36419,21259.1
23084,RABBIT NIGHT LIGHT,30788,66964.99
22492,MINI PAINT SET VINTAGE,26633,16937.82


### What percentage of orders are cancelled?

In [ ]:
%%sql
SELECT
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN InvoiceNo LIKE 'C%' THEN InvoiceNo END)
    /   COUNT(DISTINCT InvoiceNo), 2
  ) as cancellation_percentage
FROM online_retail;



 * sqlite:///online_retail.db
Done.


cancellation_percentage
14.81


### Which customers are valuable but have not ordered recently?

In [ ]:
%%sql
SELECT
CustomerId,
strftime('%Y-%m', InvoiceDate) AS last_order_date,
COUNT(DISTINCT InvoiceNo) as number_of_orders,
ROUND(SUM(UnitPrice*Quantity),2) as total_revenue
FROM online_retail
WHERE QUANTITY>0 AND UnitPrice>0 AND CustomerId is not NULL
GROUP BY CustomerId
ORDER BY total_revenue DESC, last_order_date ASC
LIMIT 20;

 * sqlite:///online_retail.db
Done.


CustomerID,last_order_date,number_of_orders,total_revenue
14646.0,2010-12,73,280206.02
18102.0,2010-12,60,259657.3
17450.0,2010-12,46,194550.79
16446.0,2011-05,2,168472.5
14911.0,2010-12,201,143825.06
12415.0,2011-01,21,124914.53
14156.0,2010-12,55,117379.63
17511.0,2010-12,31,91062.38
16029.0,2010-12,63,81024.84
12346.0,2011-01,1,77183.6


### Can we create simple customer segments?

In [ ]:
%%sql
SELECT
customer_segment,
COUNT(*) as number_of_customers
FROM (
  SELECT
  CustomerId,
  ROUND(SUM(UnitPrice*Quantity),2) as total_revenue,
  CASE
  WHEN SUM(UnitPrice * Quantity)>= 5000 THEN 'High Value'
  WHEN SUM(UnitPrice * Quantity)>= 1000 THEN 'Medium Value'
  ELSE 'Low Value'
  END as customer_segment
  FROM online_retail
  WHERE UnitPrice > 0 AND Quantity > 0 AND CustomerId is not NULL
  GROUP BY CustomerId
)as customer_segment
GROUP BY customer_segment
ORDER BY number_of_customers DESC;


 * sqlite:///online_retail.db
Done.


customer_segment,number_of_customers
Low Value,2670
Medium Value,1393
High Value,275
